## Ingest Circuits Data (CSV to Delta Lake)

This notebook reads the **circuits.csv** file (containing Formula 1 race circuit details) from the landing volume and loads it into a Bronze Delta table for downstream processing.

**Steps:**
1. **Read** the CSV file using the Spark DataFrame Reader API
2. **Enrich** the data by adding metadata columns:
   - `source_file` - the original file path (for traceability)
   - `ingestion_timestamp` - when the data was loaded (for auditing)
3. **Write** the enriched data to the Bronze Delta table `formula1.bronze.circuits`

#### Step 1: Read the CSV File
We use Spark's DataFrame Reader to load `circuits.csv` from the landing volume. This file has details about each Formula 1 circuit - name, location, and GPS coordinates.

**What's happening in the code below:**
- `format('csv')` - tells Spark the file is a CSV
- `option('header', True)` - the first row contains column names, not data
- `.schema(circutes_schema)` - we tell Spark the exact column types (defined below)
- `.load(...)` - the file path inside our volume

In [0]:
circuits_df = (
  spark.read
  .format('csv')
  .option('header', True)
 #  .option('mode', 'FAILFAST')
  # .option ('inferSchema','True')
   .schema(circutes_schema)
  .load('/Volumes/formula1/landing/files/circuits.csv'))

In [0]:
from pyspark.sql.types import *
circutes_schema = StructType([
  StructField('circuitId', StringType(), True),
  StructField('url', StringType(), True),
  StructField('circuitName', StringType(), True),
  StructField('lat', DoubleType(),True),
  StructField('long', DoubleType(), True),
 StructField('locality', StringType(), True),
  StructField('country', StringType(), True)
])


In [0]:
display(circuits_df)

#### Define an Explicit Schema
Instead of letting Spark guess the data types (which can be slow and sometimes wrong), we tell it exactly what each column should be.

**Why do we do this?**
- Faster reads - Spark doesn't need to scan the whole file to figure out types
- Correct types - `lat` and `long` are stored as decimals (Double), not text
- Early error detection - if a row has bad data, we catch it immediately

**Note:** This cell must run *before* the read cell above, since it defines the schema used there.

In [0]:
display(circutes_schema)

#### Step 2: Add Metadata Columns
We add two extra columns to the raw data for tracking purposes:
- **`ingestion_timestamp`** - records *when* this data was loaded (useful for knowing how fresh it is)
- **`source_file`** - records *which file* the row came from (useful when loading multiple files)

**Why is this important?** If something looks wrong in the data later, these columns help you trace back to *when* and *where* the data was loaded.

In [0]:
from pyspark.sql.functions import *
circuits_final_df = (
 (circuits_df
 .withColumn('ingestion_timestamp', current_timestamp())
 .withColumn('source_file', col('_metadata.file_path'))))
display(circuits_final_df)

#### Step 3: Write to Bronze Delta Table
Finally, we save the enriched DataFrame as a Delta table (`formula1.bronze.circuits`).

**What's happening:**
- `format('delta')` - saves as Delta format (supports versioning, time travel, fast queries)
- `mode('overwrite')` - replaces the entire table each time (clean reload from source)
- `saveAsTable(...)` - registers it in Unity Catalog so anyone can query it with SQL

In [0]:
(
circuits_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable('formula1.bronze.circuit') 
    
    # table name is catalog, schema and table name so u can replace it with your own predefined variables see the next page for race file 
)